# Stage B: fine-tune the encoder student (Kaggle GPU)

The Stage B procedure in `docs/WORKFLOWS.md` trains `answerdotai/ModernBERT-base`
(optionally also `microsoft/deberta-v3-small` for comparison) on the KL soft
targets from `data/interim/soft_targets.parquet`, evaluates each epoch on an
internal dev split held out from the training pool, and writes:

- `models/stage_b/` (and `models/stage_b_deberta/` if you run the optional cell):
  HF `save_pretrained` encoder + tokenizer, `head.pt` (pooling head weights),
  `metadata.json` (base model, param count, hyperparameters, dev/val metrics
  per epoch, the epoch selected).
- `preds/stage_b_{split}_logits.csv` for every split present locally (`val`
  always; `test`, `test_ood`, `test_evasion` if you attached them too).

**Expected runtime on a T4** (single GPU, `fp16=True`): a few minutes per
epoch for a training pool in the low tens of thousands of prompts; 3 epochs
plus per-epoch dev/val eval and the prediction
pass should comfortably fit in well under an hour. DeBERTa-v3-small is
similar or a little faster per epoch at the same max length.

**Before running:** attach a private Kaggle Dataset containing this repo
(so `reflex_sentry` is importable) plus `data/processed/*.parquet` and
`data/interim/soft_targets.parquet` -- see the next cell.


## 1. Prepare installation

The repository is installed with the `stage_b` extra after attaching and
copying it below. The local export/quantization pipeline is in
`scripts/run_stage_b_local.sh`; see `docs/WORKFLOWS.md`.


## 2. Attach the private dataset

In the Kaggle notebook UI: **Add Input -> Datasets -> Your Datasets**, pick
(or first create) a private dataset that contains:

- this repository (or at least `reflex_sentry/`, `pyproject.toml`, and `README.md`)
- `data/processed/train.parquet`, `data/processed/val.parquet`, and any of
  `test.parquet` / `test_ood.parquet` / `test_evasion.parquet` you have
- `data/interim/soft_targets.parquet`

It will be mounted read-only under `/kaggle/input/<dataset-slug>/`. Set
`DATASET_SLUG` below to match. If you kept the repo's own directory layout
inside the dataset, the defaults below need no change beyond the slug.


In [ ]:
import os
from pathlib import Path

DATASET_SLUG = "reflex-sentry-data"  # <-- change to your attached dataset's slug
INPUT_ROOT = Path("/kaggle/input") / DATASET_SLUG

assert INPUT_ROOT.exists(), (
    f"{INPUT_ROOT} not found -- attach your dataset and/or fix DATASET_SLUG. "
    f"Available inputs: {sorted(p.name for p in Path('/kaggle/input').glob('*'))}"
)
print("dataset root:", INPUT_ROOT)
for p in sorted(INPUT_ROOT.rglob("*.parquet")):
    print(" ", p.relative_to(INPUT_ROOT))


## 3. Install the repo

Copies the attached dataset's repo files into `/kaggle/working/reflex-sentry`
(a writable location -- `/kaggle/input` is read-only) and installs it in
editable mode with the `stage_b` extra.


In [ ]:
%%capture
import shutil
from pathlib import Path

REPO_SRC = INPUT_ROOT  # assumes the dataset root *is* the repo checkout;
                        # change to INPUT_ROOT / "reflex-sentry" if you nested it
REPO_DIR = Path("/kaggle/working/reflex-sentry")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(REPO_SRC, REPO_DIR, ignore=shutil.ignore_patterns("data", ".git"))

!pip install -q -e "{REPO_DIR}[stage_b]"


In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR))

import torch
from reflex_sentry.models import stage_b as SB

print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))


## 4. Paths

Point at the parquet files inside the attached dataset, and choose a
`/kaggle/working` output directory (Kaggle persists `/kaggle/working` to the
notebook's Output tab, which is how you download results afterwards).


In [ ]:
DATA_DIR = INPUT_ROOT / "data" / "processed"
TARGETS_PATH = INPUT_ROOT / "data" / "interim" / "soft_targets.parquet"
TRAIN_PATH = DATA_DIR / "train.parquet"
VAL_PATH = DATA_DIR / "val.parquet"

OUT_DIR = Path("/kaggle/working/models/stage_b")
PREDS_DIR = Path("/kaggle/working/preds")
PREDS_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_PATH.exists(), TRAIN_PATH
assert TARGETS_PATH.exists(), TARGETS_PATH
assert VAL_PATH.exists(), VAL_PATH

PRESENT_SPLITS = [s for s in ["val", "test", "test_ood", "test_evasion"] if (DATA_DIR / f"{s}.parquet").exists()]
print("splits present locally:", PRESENT_SPLITS)


## 5. Train ModernBERT-base

Plain PyTorch loop (`reflex_sentry.models.stage_b.train`), called directly
as a Python function rather than through the CLI so this cell's variables
stay inspectable. Checkpoint selection uses an internal dev split (10% of
the training pool, held out inside `train()`) scored by KL loss against its
soft targets -- `val` is *not* used for that; it is only logged per epoch for
your own visibility, and used later for temperature scaling and threshold
selection (see `docs/WORKFLOWS.md`), never here.


In [ ]:
metadata = SB.train(
    base="answerdotai/ModernBERT-base",
    out_dir=OUT_DIR,
    train_parquet=TRAIN_PATH,
    targets_parquet=TARGETS_PATH,
    val_parquet=VAL_PATH,          # informational logging only, see markdown above
    dev_ratio=0.1,
    max_len=256,
    epochs=3,
    lr=3e-5,
    head_lr=1e-3,
    batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    seed=13,
    grad_clip=1.0,
    fp16=torch.cuda.is_available(),
)
print("selected epoch:", metadata["best_epoch"], "dev_kl:", metadata["best_dev_kl"])
metadata


## 5b. (Optional) Also train DeBERTa-v3-small for comparison

The optional comparison base. Skip this cell if you only need one Stage B checkpoint;
it roughly doubles this notebook's runtime.


In [ ]:
TRAIN_DEBERTA = False  # flip to True to also train the alt base

if TRAIN_DEBERTA:
    deberta_out = Path("/kaggle/working/models/stage_b_deberta")
    deberta_metadata = SB.train(
        base="microsoft/deberta-v3-small",
        out_dir=deberta_out,
        train_parquet=TRAIN_PATH,
        targets_parquet=TARGETS_PATH,
        val_parquet=VAL_PATH,
        dev_ratio=0.1,
        max_len=256,
        epochs=3,
        lr=3e-5,
        head_lr=1e-3,
        batch_size=32,
        warmup_ratio=0.1,
        weight_decay=0.01,
        seed=13,
        grad_clip=1.0,
        fp16=torch.cuda.is_available(),
    )
    print("selected epoch:", deberta_metadata["best_epoch"], "dev_kl:", deberta_metadata["best_dev_kl"])


## 6. Predict: write logits CSVs for every split present

Writes `preds/stage_b_{split}_logits.csv` (id, gold, logit_safe,
logit_dangerous, logit_unsure, source, tags, latency_ms). Note that GPU
latency here is not the number that matters for this project -- the CPU
latency (batch size 1, single core) is measured later, locally, after ONNX
export and int8 quantization (`scripts/run_stage_b_local.sh`); this
notebook's `latency_ms` column is GPU timing and should not be reported as
the deployment latency.


In [ ]:
SB.predict(
    model_dir=OUT_DIR,
    splits=PRESENT_SPLITS,
    data_dir=DATA_DIR,
    out_dir=PREDS_DIR,
    batch_size=64,
)


## 7. Zip outputs for download

Kaggle's Output tab lets you download individual files or the whole working
directory, but a single zip is the least error-prone way to move a
multi-file HF checkpoint back to your machine.


In [ ]:
import shutil

shutil.make_archive("/kaggle/working/stage_b_model", "zip", OUT_DIR)
shutil.make_archive("/kaggle/working/stage_b_preds", "zip", PREDS_DIR)
print("wrote /kaggle/working/stage_b_model.zip and /kaggle/working/stage_b_preds.zip")


## 8. Next: back on your machine

1. Download `stage_b_model.zip` and `stage_b_preds.zip` from this notebook's
   Output tab.
2. Unzip `stage_b_model.zip` into `models/stage_b/` and `stage_b_preds.zip`
   into `preds/` in your local repo checkout.
3. Run `bash scripts/run_stage_b_local.sh` to export ONNX, quantize to int8,
   check fp32-vs-int8 parity on val, run the int8 CPU predictor on every
   split, and score both variants with `reflex_sentry.eval.run_all`.
